# RL-RC-DoT → VVenC: pilot trên Kaggle T4×2

Notebook kiểm tra source adapter/2 trước khi training dài. Input cần có **video MOV BDD100K Train00**. Lock65/7/100 và protocol YOLO đã nằm trong source. Chỉ thêm gói resume adapter/2 khi `MODE='resume'`; checkpoint v3/v4/DOT_PROXY cũ không tương thích.

Pilot mặc định thu thập256 tương tác, dùng hai GPU và validation rút gọn. Kết quả pilot cần được xem trước khi tạo thí nghiệm đầy đủ; không coi đây là kết quả paper.


In [ ]:
import time,sys,subprocess,json,hashlib,zipfile
from pathlib import Path
if 'RCDOT_SESSION_START_UTC' not in globals():
    RCDOT_SESSION_START_UTC=time.time()

SOURCE_COMMIT='96950935fbacf3c00a2c44f7379c9a9860f30b65'
PHASE='pilot'                 # pilot trước; full chỉ sau khi kiểm tra kết quả
MODE='new'                    # Có Input resume adapter/2: đổi thành resume
SESSION_HOURS=1.0             # Full có thể đặt11.75; gồm cả setup
SEED=20261111
GPU_IDS='0,1'
WORKERS=2
TRAIN_FRAME_LIMIT=0           # Giữ toàn bộ clip train, kể cả pilot
if PHASE=='pilot':
    ROLLOUT_STEPS=64
    ADDITIONAL_TIMESTEPS=256  # Số tương tác BỔ SUNG; SB3 làm tròn theo rollout
    VAL_VIDEOS,VAL_RATES,VAL_FRAMES=2,'50,120',64
else:
    ROLLOUT_STEPS=2048
    ADDITIONAL_TIMESTEPS=0    # 0=tiếp tục tới tổng ngân sách20M
    VAL_VIDEOS,VAL_RATES,VAL_FRAMES=7,'50,80,120,160,200',0
TOTAL_TIMESTEPS=20_000_000
REPO=Path('/kaggle/working/vvenc_rl_source')
TOOLS=Path('/kaggle/working/rl_rc_dot_tools')
RUN=Path('/kaggle/working/rcdot_'+PHASE)
assert PHASE in {'pilot','full'} and MODE in {'new','resume'}
assert 0<SESSION_HOURS<=11.75
print('CONFIG',PHASE,MODE,'WORKERS',WORKERS,'ADDITIONAL_INTERACTIONS',ADDITIONAL_TIMESTEPS)
print('SOURCE_COMMIT',SOURCE_COMMIT)
print('FULL uses a NEW run after pilot; its validation contract differs')

import torch
print("GPU_COUNT",torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count()==2,"Bật GPU T4x2 trước khi setup"


In [ ]:
def live(command,cwd=None):
    print('$',' '.join(map(str,command)),flush=True)
    p=subprocess.Popen(list(map(str,command)),cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:print(line,end='',flush=True)
        rc=p.wait()
    finally:
        if p.poll() is None:p.terminate();p.wait()
    if rc:raise RuntimeError('Command failed; inspect logs/exports in /kaggle/working')

if not (REPO/'.git').is_dir():
    live(['git','clone','--branch','feature/rl-rc-dot-vvc','--depth','1',
          'https://github.com/youngan2810/vvenc_rl.git',REPO])
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=REPO,text=True).strip(),'Tracked source modified'
live(['git','fetch','--no-tags','--depth','1','origin',SOURCE_COMMIT],REPO)
live(['git','checkout','--detach',SOURCE_COMMIT],REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==SOURCE_COMMIT
live([sys.executable,'-m','scripts.rl_rc_dot.setup_kaggle','--repo',REPO,'--source-commit',SOURCE_COMMIT,
      '--output',TOOLS,'--jobs','2','--install-system'],REPO)
toolchain=json.loads((TOOLS/'TOOLCHAIN.json').read_text())
live([sys.executable,'-c','import torch;print(torch.__version__,torch.version.cuda);print("GPU_COUNT",torch.cuda.device_count());assert torch.cuda.is_available() and torch.cuda.device_count()==2'])


In [ ]:
# Chọn đúng export mới; không tự hạ MODE=resume xuống new nếu thiếu Input.
RESUME_FROM=None
if MODE=='resume' and not (RUN/'progress.json').is_file():
    current_fingerprint=subprocess.check_output(
        [sys.executable,'-c','from scripts.rl_rc_dot.data import source_fingerprint;print(source_fingerprint())'],
        cwd=REPO,text=True).strip()
    def meta(path):
        if path.is_dir():
            return json.loads((path/'method.json').read_text()),json.loads((path/'progress.json').read_text())
        with zipfile.ZipFile(path) as z:
            if 'session_payload.bin' in z.namelist():
                with z.open('session_payload.bin') as payload,zipfile.ZipFile(payload) as inner:
                    return json.loads(inner.read('method.json')),json.loads(inner.read('progress.json'))
            return json.loads(z.read('method.json')),json.loads(z.read('progress.json'))
    candidates=[]
    inputs=Path('/kaggle/input')
    paths=list(inputs.rglob('session_payload.bin'))
    paths += [p for p in inputs.rglob('*.zip') if p.name.startswith('RL_RC_DOT_VVC_SESSION')]
    paths += [p.parent for p in inputs.rglob('export_manifest.json')]
    for path in paths:
        try:
            method,progress=meta(path)
            expected_val={'videos':VAL_VIDEOS,'rates':list(map(int,VAL_RATES.split(','))),'frames':VAL_FRAMES}
            if (method.get('schema')=='vvenc-rcdot-run/2' and method.get('source_fingerprint')==current_fingerprint
                and method.get('encoder_sha256')==toolchain['encoder_sha256'] and method.get('frames')==TRAIN_FRAME_LIMIT
                and method.get('workers')==WORKERS and method.get('rollout_steps')==ROLLOUT_STEPS
                and method.get('seed')==SEED and method.get('validation')==expected_val):
                candidates.append((progress['num_timesteps'],str(path)))
        except (ValueError,KeyError,FileNotFoundError,zipfile.BadZipFile):
            pass
    assert candidates,'Không có Input resume adapter/2 khớp source/profile/phase. Upload gói KAGGLE_SAFE mới; không dùng export v3/v4'
    candidates.sort(reverse=True)
    RESUME_FROM=Path(candidates[0][1])
    print('RESUME_SELECTED',RESUME_FROM,'OBSERVED_FRAMES',candidates[0][0])
elif MODE=='new':
    assert not RUN.exists() or not any(RUN.iterdir()),'Run đã tồn tại: đổi MODE=resume hoặc dùng Session mới'
    print('NEW_ADAPTER2_RUN — checkpoint Input không được nạp')
else:
    print('RESUME_EXISTING_WORKING_RUN',RUN)


In [ ]:
command=[sys.executable,'-u','-m','scripts.rl_rc_dot.session',
 '--input-root','/kaggle/input','--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
 '--yolo-repo',toolchain['yolo_repo'],'--weights',toolchain['weights'],'--output',RUN,
 '--mode',MODE,'--workers',str(WORKERS),'--gpu-ids',GPU_IDS,'--encoder-threads','2',
 '--seed',str(SEED),'--frames',str(TRAIN_FRAME_LIMIT),'--total-timesteps',str(TOTAL_TIMESTEPS),
 '--additional-timesteps',str(ADDITIONAL_TIMESTEPS),'--rollout-steps',str(ROLLOUT_STEPS),
 '--validation-videos',str(VAL_VIDEOS),'--validation-rates',VAL_RATES,'--validation-frames',str(VAL_FRAMES),
 '--val-every-updates','2' if PHASE=='pilot' else '25','--max-hours',str(SESSION_HOURS),
 '--session-start-utc',str(RCDOT_SESSION_START_UTC)]
if RESUME_FROM:command += ['--resume-from',str(RESUME_FROM)]
live(command,REPO)


In [ ]:
from IPython.display import display,FileLink
import shutil
progress=json.loads((RUN/'progress.json').read_text())
print('PROGRESS',json.dumps(progress,indent=2))
exports=sorted((RUN/'exports').glob('*_KAGGLE_SAFE.zip'),key=lambda p:p.stat().st_mtime)
assert exports,'Chưa có export; kiểm tra failure.json/logs'
safe=Path('/kaggle/working/RL_RC_DOT_VVC_LATEST_KAGGLE_SAFE.zip')
shutil.copyfile(exports[-1],safe)
display(FileLink(str(safe)))
print('Giữ lại toàn bộ export của từng phiên; gói SAFE này là Input resume của cùng phase/source/config')
print('Pilot chưa chứng minh hội tụ hoặc tốt hơn VVenC; gửi kết quả trước khi chạy full')
